
这是一个非常棒的实战需求。为了让你清晰地理解两者的区别，我将分别演示如何实现同一个目标：创建一个“联网搜索 Agent”。

我们将使用的工具核心都是 DuckDuckGo。

方式一：LangChain 方式（封装度高，代码少，开箱即用）。

方式二：OpenAI 原生方式（底层逻辑，需要手动处理 JSON 和函数调用循环）。


环境准备
你需要安装以下 Python 库：

pip install langchain langchain-openai langchain-community openai

pip install -U duckduckgo-search




## 方式一：使用 LangChain 实现 (High-Level)

LangChain 最大的优势是它帮我们把“定义工具”、“Prompt 组装”、“解析大模型输出”、“执行工具”、“回传结果”这一整套复杂的流程封装成了简单的 API。

这里我们使用当前最推荐的 create_tool_calling_agent 方法。

In [3]:
import dotenv
import os
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_openai import ChatOpenAI
from langchain.agents import create_tool_calling_agent, AgentExecutor
from langchain_core.prompts import ChatPromptTemplate

dotenv.load_dotenv()

os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY")
os.environ['OPENAI_BASE_URL'] = os.getenv("OPENAI_BASE_URL")

# 1. 初始化大模型
# 定义LLM模型
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# 2. 创建/加载工具(Tool)
# Langchain 内置了对 DuckDuckGo 的封装 直接实例化
search_tool = DuckDuckGoSearchRun()
tools = [search_tool]

# 3. 创建 Prompt 提示词
# 我们需要告诉 AI 它的角色 并预留 {agent_scratchpad} 位置给中间思考过程
prompt = ChatPromptTemplate.from_messages([
    ("system", "你是一个乐于助人的 AI 助手。如果需要，你可以使用搜索工具来回答问题。"),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}")  # 关键：这里存放 AI 的思考、工具调用和工具结果

])

# 4. 创建Agent 大脑
# create_tool_calling_agent 会自动把 tools 转换成 OpenAI 的 Function 格式绑定给 LLM
agent = create_tool_calling_agent(llm, tools, prompt)

# 5. 创建 AgentExecutor 执行期
# 它负责运行 Agent，拦截 Agent 的工具调用请求，执行工具，再把结果喂回给 Agent
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

# 6. 运行
print("=== LangChain Agent 运行开始 ===")
response = agent_executor.invoke({"input": "成龙是谁"})
print(f"最终答案: {response['output']}")

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


=== LangChain Agent 运行开始 ===


/Users/dingchuan/Documents/Repos/ai-demo/.venv/lib/python3.12/site-packages/langchain_core/load/serializable.py:275: PydanticDeprecatedSince211: Accessing the 'model_fields' attribute on the instance is deprecated. Instead, you should access this attribute from the model class. Deprecated in Pydantic V2.11 to be removed in V3.0.
  field = inst.model_fields.get(key)
/Users/dingchuan/Documents/Repos/ai-demo/.venv/lib/python3.12/site-packages/langchain_core/load/serializable.py:275: PydanticDeprecatedSince211: Accessing the 'model_fields' attribute on the instance is deprecated. Instead, you should access this attribute from the model class. Deprecated in Pydantic V2.11 to be removed in V3.0.
  field = inst.model_fields.get(key)
/Users/dingchuan/Documents/Repos/ai-demo/.venv/lib/python3.12/site-packages/langchain_core/load/serializable.py:200: PydanticDeprecatedSince211: Accessing the 'model_fields' attribute on the instance is deprecated. Instead, you should access this attribute from th

成龙（Jackie Chan），原名陈港生，1954年4月7日出生于中国香港，是一位著名的武打演员、导演、制片人和歌手。他以其独特的武术风格、幽默感和高难度的特技表演而闻名于世。成龙的电影作品涵盖了动作、喜剧和冒险等多种类型，代表作包括《醉拳》、《警察故事》、《红番区》和《尖峰时刻》等。

成龙不仅在亚洲电影界享有盛誉，还在好莱坞取得了巨大的成功。他的电影常常融合了中国传统武术和西方动作片的元素，深受观众喜爱。此外，成龙还积极参与慈善事业，并在2004年被联合国任命为“和平使者”。

> Finished chain.
最终答案: 成龙（Jackie Chan），原名陈港生，1954年4月7日出生于中国香港，是一位著名的武打演员、导演、制片人和歌手。他以其独特的武术风格、幽默感和高难度的特技表演而闻名于世。成龙的电影作品涵盖了动作、喜剧和冒险等多种类型，代表作包括《醉拳》、《警察故事》、《红番区》和《尖峰时刻》等。

成龙不仅在亚洲电影界享有盛誉，还在好莱坞取得了巨大的成功。他的电影常常融合了中国传统武术和西方动作片的元素，深受观众喜爱。此外，成龙还积极参与慈善事业，并在2004年被联合国任命为“和平使者”。


### LangChain 方式总结：
定义工具： 直接 DuckDuckGoSearchRun()，无需写 JSON Schema。

调用流程： AgentExecutor 自动处理了“思考-调用-执行-反馈”的死循环，你只需要调 invoke。


## 方式二：使用 OpenAI 原生 API 实现 (Low-Level)

使用原生 API，你需要亲自扮演 LangChain 的角色：你需要自己定义工具的 JSON 描述，自己解析 LLM 返回的 tool_calls，自己运行 Python 函数，然后再发起第二次请求把结果给 LLM。

In [9]:
import os
from openai import OpenAI

# 创建 client
client = OpenAI(
    api_key=os.environ["OPENAI_API_KEY"],
    base_url=os.environ["OPENAI_BASE_URL"])

# 测试调用
resp = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "user", "content": "hello"}
    ]
)

print(resp.choices[0].message.content)

Hello! How can I assist you today?


In [14]:
import json
from openai import OpenAI
from duckduckgo_search import DDGS

client = OpenAI(
    api_key=os.environ["OPENAI_API_KEY"],
    base_url=os.environ["OPENAI_BASE_URL"])


# 1. 第一步 定义具体的 Python 函数
def run_search(query):
    """实际执行搜索的函数"""
    print(f"\n[System] 正在调用 DuckDuckGo (DDGS) 搜索: {query} ...")

    # 【核心修正 2】使用 DDGS 类和上下文管理器
    try:
        with DDGS() as ddgs:
            # .text() 是新版获取搜索结果的方法
            # max_results 限制返回数量
            results = list(ddgs.text(keywords=query, max_results=2))

            if results:
                # 结果通常包含 'title', 'href', 'body'
                return json.dumps(results[0], ensure_ascii=False)
            return "未找到相关结果。"
    except Exception as e:
        return f"搜索出错: {str(e)}"


# 第二步 定义工具的 JSON Schema 给 LLM 看菜单
# 这就是 LangChain 帮我们在后台自动生成的东西
tools = [
    {
        "type": "function",
        "function": {
            "name": "run_search",
            "description": "当用户询问实时信息或你不知道的知识时，使用此工具搜索互联网。",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "用于搜索引擎的查询关键词",
                    }
                },
                "required": ["query"],
            },
        }
    }
]


# 3. 第三步 手动实现 Agent 循环
def run_openai_agent(user_query):
    # 1. 初始化消息列表
    messages = [{"role": "user", "content": user_query}]

    # 2. 第一次调用LLM
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        tools=tools,
        tool_choice="auto",
    )

    response_message = response.choices[0].message
    # 3. 检查 LLM 是否想调用工具
    tool_calls = response_message.tool_calls

    if tool_calls:
        # LLM 决定调用工具，我们先把 LLM 的这个回复加入历史记录
        messages.append(response_message)

        # 4. 解析并执行工具
        for tool_call in tool_calls:
            function_name = tool_call.function.name
            function_args = json.loads(tool_call.function.arguments)

            if function_name == "run_search":
                # 执行真正的 Python 代码
                function_response = run_search(function_args.get("query"))

                # 5. 将工具运行结果构造成 message，塞回给 LLM
                messages.append(
                    {
                        "tool_call_id": tool_call.id,  # 必须带上 ID 用于匹配
                        "role": "tool",
                        "name": function_name,
                        "content": function_response,
                    }
                )

        # 6. 第二次调用 LLM (带着工具的结果)
        # 此时 LLM 拥有了：用户问题 + 它的调用请求 + 工具的返回结果
        print("\n[System] 将搜索结果反馈给 LLM，生成最终回答...")
        final_response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
        )
        return final_response.choices[0].message.content
    else:
        # LLM 没调用工具，直接返回内容
        return response_message.content
# 运行
print("=== OpenAI Native Agent 运行开始 ===")
result = run_openai_agent("LangChain 是什么？")
print(f"最终答案: {result}")

=== OpenAI Native Agent 运行开始 ===

[System] 正在调用 DuckDuckGo (DDGS) 搜索: LangChain 是什么 ...


/var/folders/gd/xcfqj9752391g13gs68sxcfr0000gn/T/ipykernel_78833/3680337005.py:17: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:



[System] 将搜索结果反馈给 LLM，生成最终回答...
最终答案: LangChain 是一个框架，旨在简化和增强与大语言模型（如 GPT 系列模型）进行交互的过程。它主要分为六个模块，分别负责以下功能：

1. **模型管理**：管理输入和输出的过程。
2. **外部数据接入**：支持从外部数据源获取信息。
3. **链的概念**：将多个处理步骤连接成链，以实现复杂任务。
4. **上下文记忆存储/**管理：处理上下文信息的存储。
5. **智能代理**：通过智能代理执行任务。
6. **回调系统**：提供对事件响应的支持。

LangChain 使得开发者能够更轻松地搭建复杂的应用程序，利用大语言模型的强大功能。
